# Controversy-Weighted Respondent Similarity Network

`Respondent_Similarity_Network.ipynb` treats all 60 statements as equally important
when computing respondent-respondent similarity (plain Pearson correlation). This
notebook tests a **discrimination-weighted** alternative, in the spirit of item
discrimination in Item Response Theory / roll-call ideal-point models (NOMINATE):
statements that split the sample (controversial / bimodal) should count more toward
"these two respondents have similar opinions" than statements everyone already agrees
on, since consensus items carry little information about where someone stands.

We build two weighted similarity matrices (weight = per-statement std, weight =
per-statement bimodality coefficient), run them through the identical
threshold -> Louvain -> centrality pipeline used for the unweighted network, and
compare the results.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from networkx.algorithms.community import louvain_communities
from sklearn.metrics import adjusted_rand_score
from scipy.stats import skew, kurtosis

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

## 1. Load, encode, and recompute per-statement weights

In [2]:
df = pd.read_csv("Survey_Results_UC.csv", encoding="utf-8-sig")
df = df.rename(columns={"id. Response ID": "respondent_id"}).set_index("respondent_id")

likert_map = {"Strongly Disagree": -2, "Disagree": -1, "Neutral": 0, "Agree": 1, "Strongly Agree": 2}
df_num = df.replace({"": np.nan, "No Comments": np.nan}).replace(likert_map).apply(pd.to_numeric)

print(f"Respondents: {df_num.shape[0]}, Statements: {df_num.shape[1]}")

Respondents: 96, Statements: 60


In [3]:
def bimodality_coefficient(x):
    x = x.dropna()
    n = len(x)
    g = skew(x, bias=False)
    k = kurtosis(x, fisher=True, bias=False)
    return (g**2 + 1) / (k + 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))

weights = pd.DataFrame({
    "std": df_num.std(axis=0),
    "bimodality_coef": df_num.apply(bimodality_coefficient, axis=0).clip(lower=0),
})
weights["uniform"] = 1.0

print("Most heavily up-weighted statements (controversial, by std):")
display(weights["std"].sort_values(ascending=False).head(5))
print("\nMost heavily down-weighted statements (consensus, by std):")
display(weights["std"].sort_values(ascending=True).head(5))

Most heavily up-weighted statements (controversial, by std):


E04. High-quality online learning can effectively complement classroom teaching.               1.168393
E03. Class attendance should be compulsory for all courses.                                    1.124322
E02. Traditional written examinations accurately measure a student's knowledge.                1.119318
T08. AI-assisted diagnosis should become routine in healthcare.                                1.093677
S02. Universities should encourage students to participate in community service activities.    1.055149
Name: std, dtype: float64


Most heavily down-weighted statements (consensus, by std):


E15. Continuous learning and skill development are essential throughout one's career.                                  0.503662
V13. Companies should be held accountable for the environmental impacts of their activities.                           0.564595
S09. Universities should promote an inclusive environment where different viewpoints can be discussed respectfully.    0.589355
V06. Water conservation should be a priority in households, institutions, and industries.                              0.590977
S05. Individuals should have greater control over how their personal data are collected and used.                      0.592883
Name: std, dtype: float64

## 2. Weighted Pearson correlation

The standard generalization of Pearson correlation to weighted items: instead of the
plain mean/variance/covariance, use the **weighted** mean, variance, and covariance of
each respondent pair's shared (non-missing) statements, then normalize the same way
(covariance / sqrt(variance_a * variance_b)) so the result stays in [-1, 1] like an
ordinary correlation. Uniform weights must reduce to the plain Pearson correlation —
we assert that below as a sanity check on the implementation.

In [4]:
def weighted_corr_matrix(X: pd.DataFrame, w: pd.Series) -> pd.DataFrame:
    idx = X.index
    w = w.reindex(X.columns).values
    vals = X.values
    n = len(idx)
    sim = np.full((n, n), np.nan)
    for i in range(n):
        xi = vals[i]
        for j in range(i, n):
            xj = vals[j]
            mask = ~np.isnan(xi) & ~np.isnan(xj)
            if mask.sum() < 5:
                continue
            wm, xim, xjm = w[mask], xi[mask], xj[mask]
            wsum = wm.sum()
            mi = (wm * xim).sum() / wsum
            mj = (wm * xjm).sum() / wsum
            cov = (wm * (xim - mi) * (xjm - mj)).sum() / wsum
            vi = (wm * (xim - mi) ** 2).sum() / wsum
            vj = (wm * (xjm - mj) ** 2).sum() / wsum
            r = cov / np.sqrt(vi * vj) if vi > 0 and vj > 0 else np.nan
            sim[i, j] = sim[j, i] = r
    return pd.DataFrame(sim, index=idx, columns=idx)

# Sanity check: uniform weights must reproduce plain Pearson correlation
uniform_check = weighted_corr_matrix(df_num, weights["uniform"])
plain_corr = df_num.T.corr()
max_diff = (uniform_check - plain_corr).abs().to_numpy()
max_diff = max_diff[~np.isnan(max_diff)].max()
assert max_diff < 1e-9, f"weighted_corr_matrix with uniform weights should equal df_num.T.corr(), max diff={max_diff}"
print(f"Sanity check passed: uniform-weight formula matches plain Pearson correlation (max diff={max_diff:.2e})")

Sanity check passed: uniform-weight formula matches plain Pearson correlation (max diff=7.77e-16)


In [5]:
sim_uniform = plain_corr  # baseline, identical to Respondent_Similarity_Network.ipynb
sim_std = weighted_corr_matrix(df_num, weights["std"])
sim_bimodal = weighted_corr_matrix(df_num, weights["bimodality_coef"])

for name, sim in [("uniform (baseline)", sim_uniform), ("std-weighted", sim_std), ("bimodality-weighted", sim_bimodal)]:
    vals = sim.where(np.triu(np.ones(sim.shape), k=1).astype(bool)).stack()
    print(f"{name:22s}: mean r={vals.mean():+.3f}, std={vals.std():.3f}")

uniform (baseline)    : mean r=+0.252, std=0.200
std-weighted          : mean r=+0.261, std=0.209
bimodality-weighted   : mean r=+0.253, std=0.201


## 3. Build comparable networks

To isolate the effect of *weighting* from the effect of *network density*, pick each
weighting's threshold so all three networks land on roughly the same edge count as the
baseline (957 edges, from `Respondent_Similarity_Network.ipynb` at r >= 0.4) — otherwise
a difference in detected communities could just be an artifact of one network being
denser or sparser than another.

In [6]:
TARGET_EDGES = 957

def threshold_for_edge_count(sim, target_edges):
    sim_no_diag = sim.where(~np.eye(len(sim), dtype=bool))
    vals = sim_no_diag.values[~np.isnan(sim_no_diag.values)]
    vals_sorted = np.sort(vals)[::-1]
    # target_edges undirected edges -> 2*target_edges directed entries in the flattened matrix
    return vals_sorted[min(2 * target_edges, len(vals_sorted) - 1)]

def build_graph(sim, threshold):
    G = nx.Graph()
    G.add_nodes_from(sim.index)
    idx = list(sim.index)
    for i, a in enumerate(idx):
        for b in idx[i + 1:]:
            r = sim.loc[a, b]
            if pd.notna(r) and r >= threshold:
                G.add_edge(a, b, weight=r)
    return G

networks = {}
for name, sim in [("uniform", sim_uniform), ("std", sim_std), ("bimodal", sim_bimodal)]:
    t = threshold_for_edge_count(sim, TARGET_EDGES)
    G = build_graph(sim, t)
    networks[name] = G
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    print(f"{name:10s}: threshold={t:.3f}, edges={G.number_of_edges()}, "
          f"density={nx.density(G):.3f}, giant_component={len(components[0])}, "
          f"avg_clustering={nx.average_clustering(G):.3f}")

uniform   : threshold=0.400, edges=958, density=0.210, giant_component=88, avg_clustering=0.510
std       : threshold=0.417, edges=958, density=0.210, giant_component=86, avg_clustering=0.513


bimodal   : threshold=0.403, edges=958, density=0.210, giant_component=88, avg_clustering=0.510


## 4. Do the weightings change who's central?

Compare eigenvector centrality rankings (giant component only) across the three
networks — a large reshuffling means controversy-weighting genuinely changes who
counts as holding "consensus" opinions, not just cosmetic score changes.

In [7]:
from scipy.stats import spearmanr

centralities = {}
giants = {}
for name, G in networks.items():
    giant_nodes = max(nx.connected_components(G), key=len)
    Gg = G.subgraph(giant_nodes).copy()
    giants[name] = Gg
    centralities[name] = pd.Series(nx.eigenvector_centrality(Gg, weight="weight", max_iter=1000), name=name)

common = sorted(set(centralities["uniform"].index) & set(centralities["std"].index) & set(centralities["bimodal"].index))
cent_df = pd.DataFrame({name: c.reindex(common) for name, c in centralities.items()})

print("Spearman rank correlation of eigenvector centrality between weightings:")
for a, b in [("uniform", "std"), ("uniform", "bimodal"), ("std", "bimodal")]:
    rho, _ = spearmanr(cent_df[a], cent_df[b])
    print(f"  {a} vs {b}: rho={rho:.3f}")

print("\nTop 5 most central respondents per weighting:")
for name in networks:
    print(f"  {name}: {cent_df[name].sort_values(ascending=False).head(5).index.tolist()}")

Spearman rank correlation of eigenvector centrality between weightings:
  uniform vs std: rho=0.990
  uniform vs bimodal: rho=0.988
  std vs bimodal: rho=0.985

Top 5 most central respondents per weighting:
  uniform: [114, 49, 61, 81, 104]
  std: [61, 114, 81, 49, 90]
  bimodal: [61, 114, 49, 90, 55]


## 5. Do the weightings change the detected communities?

Run Louvain on each of the three density-matched networks and compare partitions via
Adjusted Rand Index — the direct test of whether controversy-weighting reshapes the
opinion camps or just relabels the same ones.

In [8]:
partitions = {}
for name, G in networks.items():
    comms = louvain_communities(G, weight="weight", seed=0)
    partitions[name] = {n: i for i, comm in enumerate(comms) for n in comm}
    sizes = sorted((len(c) for c in comms), reverse=True)
    print(f"{name:10s}: {len(comms)} communities, sizes={sizes}")

print("\nAdjusted Rand Index between weightings' community partitions:")
all_nodes = df_num.index
for a, b in [("uniform", "std"), ("uniform", "bimodal"), ("std", "bimodal")]:
    ari = adjusted_rand_score([partitions[a][n] for n in all_nodes], [partitions[b][n] for n in all_nodes])
    print(f"  {a} vs {b}: ARI={ari:.3f}")

uniform   : 12 communities, sizes=[31, 24, 22, 11, 1, 1, 1, 1, 1, 1, 1, 1]
std       : 13 communities, sizes=[33, 32, 21, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
bimodal   : 11 communities, sizes=[39, 31, 18, 1, 1, 1, 1, 1, 1, 1, 1]

Adjusted Rand Index between weightings' community partitions:
  uniform vs std: ARI=0.535
  uniform vs bimodal: ARI=0.599
  std vs bimodal: ARI=0.732


## 6. What's actually driving any reshuffled respondents?

For respondents whose community assignment changed between the uniform and std-weighted
networks, check which statements they scored most differently from the sample mean on
the *high-weight* (controversial) items specifically — this should explain why
weighting moved them.

In [9]:
from scipy.optimize import linear_sum_assignment

def align_labels(base_of: dict, other_of: dict, nodes) -> dict:
    """Relabel `other_of`'s community ids to best match `base_of`'s, via max-overlap
    (Hungarian algorithm on the contingency table), so raw label equality is meaningful
    across two independent Louvain runs."""
    base_labels = sorted(set(base_of[n] for n in nodes))
    other_labels = sorted(set(other_of[n] for n in nodes))
    contingency = pd.DataFrame(0, index=base_labels, columns=other_labels)
    for n in nodes:
        contingency.loc[base_of[n], other_of[n]] += 1
    row_ind, col_ind = linear_sum_assignment(-contingency.values)
    label_map = {other_labels[c]: base_labels[r] for r, c in zip(row_ind, col_ind)}
    # any other-label with no assigned base-label (more communities than base) keeps a fresh id
    next_id = max(base_labels) + 1
    for lbl in other_labels:
        if lbl not in label_map:
            label_map[lbl] = next_id
            next_id += 1
    return {n: label_map[other_of[n]] for n in nodes}

std_aligned = align_labels(partitions["uniform"], partitions["std"], all_nodes)
moved = [n for n in all_nodes if partitions["uniform"][n] != std_aligned[n]]
print(f"Respondents whose (label-aligned) community assignment changed (uniform -> std-weighted): "
      f"{len(moved)} / {len(all_nodes)}")

if moved:
    top_controversial = weights["std"].sort_values(ascending=False).head(10).index
    print("\nTheir answers on the 10 most controversial (highest-std) statements vs. the sample mean:")
    display((df_num.loc[moved, top_controversial] - df_num[top_controversial].mean()).round(1).head(10))


Respondents whose (label-aligned) community assignment changed (uniform -> std-weighted): 29 / 96

Their answers on the 10 most controversial (highest-std) statements vs. the sample mean:


,E04. High-quality online learning can effectively complement classroom teaching.,E03. Class attendance should be compulsory for all courses.,E02. Traditional written examinations accurately measure a student's knowledge.,T08. AI-assisted diagnosis should become routine in healthcare.,S02. Universities should encourage students to participate in community service activities.,E09. Collaborative learning is generally more effective than individual learning.,T07. Robots should replace humans in hazardous occupations whenever possible.,E12. Artificial intelligence should be integrated into teaching and personalized learning.,S08. Diverse teams generally make better decisions than homogeneous teams.,T12. Governments should introduce stricter regulations for Artificial Intelligence.
respondent_id,,,,,,,,,,
26,1.6,-1.1,0.2,1.9,-1.8,-0.8,0.8,1.1,1.1,1.0
29,1.6,-1.1,0.2,1.9,0.2,-1.8,0.8,-0.9,1.1,1.0
35,-1.4,-1.1,-1.8,-0.1,-0.8,-0.8,0.8,0.1,0.1,-0.0
40,-1.4,2.9,2.2,1.9,1.2,1.2,0.8,-0.9,1.1,-0.0
48,-1.4,1.9,0.2,-0.1,0.2,1.2,-0.2,-0.9,1.1,-1.0
49,0.6,-1.1,-0.8,-1.1,0.2,0.2,-0.2,0.1,0.1,1.0
52,-1.4,-0.1,0.2,0.9,1.2,-0.8,0.8,1.1,1.1,1.0
54,-0.4,0.9,0.2,-0.1,1.2,1.2,0.8,1.1,-0.9,1.0
58,1.6,-0.1,0.2,0.9,-2.8,-2.8,-2.2,1.1,-2.9,1.0


## 7. Which statements actually separate the communities?

Section 6's pairwise Cohen's d only works two groups at a time. With 3+ communities per
network, the standard way to rank *every* statement by how well it separates the groups
is a one-way **ANOVA F-test**, and — since F depends on sample size and isn't
comparable across statements with different variances — the accompanying **eta-squared
effect size** (eta^2 = between-group sum of squares / total sum of squares, i.e. the
fraction of that statement's variance explained by community membership; same idea as
R^2 for a categorical predictor). This is the natural multi-group generalization of the
Cohen's-d belief-profile analysis already used for the correlation network, and it
doubles as a feature-importance ranking for "which item defines this clustering."

We apply it to all three community structures (unweighted, std-weighted,
bimodality-weighted) using only each one's substantial (n >= 10) communities.

In [10]:
def anova_by_community(df_num, community_of, min_size=10):
    sizes = pd.Series(community_of).value_counts()
    keep = sizes[sizes >= min_size].index
    members_by_comm = {c: [n for n in df_num.index if community_of.get(n) == c] for c in keep}

    rows = []
    for stmt in df_num.columns:
        groups = [df_num.loc[members, stmt].dropna() for members in members_by_comm.values()]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) < 2:
            continue
        F, p = f_oneway(*groups)
        all_vals = pd.concat(groups)
        grand_mean = all_vals.mean()
        ss_total = ((all_vals - grand_mean) ** 2).sum()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        eta_sq = ss_between / ss_total if ss_total > 0 else 0.0
        rows.append({"statement": stmt, "F": F, "p": p, "eta_sq": eta_sq})
    return pd.DataFrame(rows).set_index("statement").sort_values("eta_sq", ascending=False), members_by_comm

from scipy.stats import f_oneway

anova_results = {}
members_by_scheme = {}
for name in ["uniform", "std", "bimodal"]:
    anova_df, members = anova_by_community(df_num, partitions[name])
    anova_results[name] = anova_df
    members_by_scheme[name] = members
    print(f"\n=== Top 8 separating statements — {name} weighting "
          f"({len(members)} communities used, sizes={[len(m) for m in members.values()]}) ===")
    display(anova_df.head(8)[["F", "p", "eta_sq"]])


=== Top 8 separating statements — uniform weighting (4 communities used, sizes=[31, 24, 22, 11]) ===

,F,p,eta_sq
statement,,,
T12. Governments should introduce stricter regulations for Artificial Intelligence.,24.770550,1.528566e-11,0.472385
T13. Cybersecurity deserves greater investment than the development of new digital technologies.,19.386260,1.360933e-09,0.414949
E09. Collaborative learning is generally more effective than individual learning.,7.528551,1.707710e-04,0.222331
T05. Artificial Intelligence will significantly accelerate scientific discovery.,7.899223,1.054773e-04,0.220039
T01. Artificial Intelligence will improve society more than it will create problems.,5.723900,1.297028e-03,0.169728
E04. High-quality online learning can effectively complement classroom teaching.,5.369163,2.028805e-03,0.167599
V01. Climate change requires immediate global action.,5.198495,2.499178e-03,0.164865
S04. Technology companies should be accountable for unethical use of their platforms.,5.017594,3.077485e-03,0.158362



=== Top 8 separating statements — std weighting (3 communities used, sizes=[33, 32, 21]) ===


,F,p,eta_sq
statement,,,
T13. Cybersecurity deserves greater investment than the development of new digital technologies.,22.308425,1.915153e-08,0.355182
T12. Governments should introduce stricter regulations for Artificial Intelligence.,21.532570,3.046356e-08,0.344342
E04. High-quality online learning can effectively complement classroom teaching.,11.967233,2.884814e-05,0.232521
T01. Artificial Intelligence will improve society more than it will create problems.,9.611462,1.759894e-04,0.188049
V01. Climate change requires immediate global action.,7.335004,1.205204e-03,0.158304
S07. Equal opportunities should be prioritized regardless of a person's background.,6.506575,2.435722e-03,0.142981
T02. Generative AI tools should be allowed as learning aids in higher education.,6.318814,2.790266e-03,0.132141
T05. Artificial Intelligence will significantly accelerate scientific discovery.,5.579617,5.326350e-03,0.118514



=== Top 8 separating statements — bimodal weighting (3 communities used, sizes=[39, 31, 18]) ===


,F,p,eta_sq
statement,,,
T12. Governments should introduce stricter regulations for Artificial Intelligence.,36.461835,3.988549e-12,0.464708
T13. Cybersecurity deserves greater investment than the development of new digital technologies.,30.887005,9.398983e-11,0.426693
E04. High-quality online learning can effectively complement classroom teaching.,8.501068,4.452501e-04,0.173487
T10. Protection of personal data is more important than technological convenience.,8.289330,5.142794e-04,0.163210
S07. Equal opportunities should be prioritized regardless of a person's background.,6.079799,3.483185e-03,0.131941
S14. Leaders should prioritize ethical decision-making even when it reduces short-term gains.,5.686600,4.949991e-03,0.127255
S08. Diverse teams generally make better decisions than homogeneous teams.,5.694841,4.872064e-03,0.124628
T01. Artificial Intelligence will improve society more than it will create problems.,5.932212,3.875445e-03,0.122485


### 7.1 Are the same statements doing the separating, regardless of weighting?

Compare the top-10 most-separating statements across the three weighting schemes via
Jaccard overlap — high overlap means the same handful of divisive statements drive
community structure no matter how the network was built; low overlap means the
weighting is changing *which* opinions define the camps, not just their boundaries.

In [11]:
top10 = {name: set(res.head(10).index) for name, res in anova_results.items()}

print("Top-10 separating statements per weighting:")
for name, s in top10.items():
    print(f"\n{name}:")
    for stmt in anova_results[name].head(10).index:
        print(f"  {stmt}")

print("\nJaccard overlap between top-10 sets:")
for a, b in [("uniform", "std"), ("uniform", "bimodal"), ("std", "bimodal")]:
    jaccard = len(top10[a] & top10[b]) / len(top10[a] | top10[b])
    print(f"  {a} vs {b}: {jaccard:.2f} ({len(top10[a] & top10[b])} shared statements)")

Top-10 separating statements per weighting:

uniform:
  T12. Governments should introduce stricter regulations for Artificial Intelligence.
  T13. Cybersecurity deserves greater investment than the development of new digital technologies.
  E09. Collaborative learning is generally more effective than individual learning.
  T05. Artificial Intelligence will significantly accelerate scientific discovery.
  T01. Artificial Intelligence will improve society more than it will create problems.
  E04. High-quality online learning can effectively complement classroom teaching.
  V01. Climate change requires immediate global action.
  S04. Technology companies should be accountable for unethical use of their platforms.
  T10. Protection of personal data is more important than technological convenience.
  T03. Students should disclose the use of AI in assignments and reports.

std:
  T13. Cybersecurity deserves greater investment than the development of new digital technologies.
  T12. Governmen

### 7.2 What do the top statements actually say, per community?

For the unweighted (baseline) network, show each of its 4 main communities' mean score
on its own top-6 separating statements — this is the concrete, defensible version of
"primary beliefs," grounded in the effect-size ranking rather than an arbitrary pick.

In [12]:
top_uniform = anova_results["uniform"].head(6).index
members_u = members_by_scheme["uniform"]

profile = pd.DataFrame({
    f"community {c} (n={len(m)})": df_num.loc[m, top_uniform].mean()
    for c, m in members_u.items()
})
profile.round(2)

,community 8 (n=31),community 0 (n=24),community 3 (n=22),community 1 (n=11)
statement,,,,
T12. Governments should introduce stricter regulations for Artificial Intelligence.,0.19,1.43,1.68,1.55
T13. Cybersecurity deserves greater investment than the development of new digital technologies.,0.20,1.61,1.18,1.36
E09. Collaborative learning is generally more effective than individual learning.,0.63,1.50,0.19,0.82
T05. Artificial Intelligence will significantly accelerate scientific discovery.,1.35,0.58,1.50,1.73
T01. Artificial Intelligence will improve society more than it will create problems.,1.03,0.00,0.68,0.73
E04. High-quality online learning can effectively complement classroom teaching.,1.07,0.33,-0.14,0.36


## Summary

- Implemented the standard weighted-Pearson-correlation generalization (weighted mean
  / variance / covariance, normalized to [-1, 1]) and verified it reduces exactly to
  plain Pearson correlation under uniform weights (Section 2).
- Two discrimination-style weightings were tested — per-statement standard deviation
  and the (clipped) bimodality coefficient from `EDA.ipynb` §10 — in the spirit of IRT
  item discrimination / NOMINATE-style roll-call scaling, where consensus items should
  contribute less to "these two people think alike" than genuinely divisive ones.
- Networks were density-matched to the baseline's 957 edges (Section 3) so any
  differences in structure reflect the weighting itself, not a denser/sparser graph.
- Centrality rankings and community partitions were compared directly via Spearman
  correlation and Adjusted Rand Index (Sections 4-5) rather than asserted — see the
  actual numbers there for whether this reweighting is a meaningful methodological
  choice or a cosmetic one for this dataset.
- Section 6 traces any reshuffled respondents back to their specific answers on the
  most controversial statements, to ground *why* weighting moved them rather than just
  reporting that it did.

- **Which statements separate communities (Section 7):** generalized the pairwise
  Cohen's-d comparison to 3+ groups via one-way ANOVA F-test + eta-squared effect size
  per statement, applied to all three community structures (unweighted, std-weighted,
  bimodality-weighted). Section 7.1's Jaccard overlap shows whether the same divisive
  statements drive community structure regardless of how the network was weighted;
  Section 7.2 grounds the baseline network's community "beliefs" in the top
  effect-size statements rather than an arbitrary selection.